# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

CUDA available: True
GPU: Tesla T4
Thu Jul 16 04:04:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/APS360/Final_Project


## 3. Install Dependencies

In [3]:
%pip install -r requirements.txt

Ignoring pywin32: markers 'platform_system == "Windows"' don't match your environment


## 4. Verify Or Prepare Expert-Labelled Data

Download L2DTnH's `2_16000_chatlogs_english_only.csv` to `data/l2dtnh/l2dtnh_english.csv`. The preparation script normalizes the messages, removes contradictory normalized labels, and creates match-grouped train/validation/test splits.

In [4]:
from pathlib import Path
import subprocess

import pandas as pd

raw_path = Path('data/l2dtnh/l2dtnh_english.csv')
prepared_path = Path('data/l2dtnh/l2dtnh_prepared.csv')

if not raw_path.exists():
    raise FileNotFoundError(
        'Upload 2_16000_chatlogs_english_only.csv as data/l2dtnh/l2dtnh_english.csv.'
    )

# Always rebuild so grouped split assignments and the audit match the current code.
subprocess.run(['python', 'prepare_l2dtnh.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(pd.crosstab(df['split'], df['label']))
print('Groups per split:', df.groupby('split')['group_id'].nunique().to_dict())

                                          raw_text  \
0                            olaf is bronze hahaha   
1  yes i know kayle is too but u gyus are premades   
2                                            kayle   
3                                if u dont die top   
4                    kayle plays like a challenger   

                                              text  label  group_id split  
0                            olaf is bronze hahaha      1      1009  test  
1  yes i know kayle is too but u gyus are premades      1      1009  test  
2                                            kayle      0      1009  test  
3                                if u dont die top      0      1009  test  
4                    kayle plays like a challenger      0      1009  test  
label     0    1
split           
test   1939  195
train  9123  927
val    1911  198
Groups per split: {'test': 16, 'train': 70, 'val': 15}


## 5. Train LSTM

In [5]:
!python train.py

Device: cuda
Epoch 01/10 | train loss 1.0367 bal_acc 0.715 f1 0.302 | val loss 0.8040 bal_acc 0.787 f1 0.416  <- saved
Epoch 02/10 | train loss 0.6864 bal_acc 0.834 f1 0.500 | val loss 0.7077 bal_acc 0.817 f1 0.490  <- saved
Epoch 03/10 | train loss 0.4755 bal_acc 0.894 f1 0.612 | val loss 0.7714 bal_acc 0.818 f1 0.558
Epoch 04/10 | train loss 0.3305 bal_acc 0.929 f1 0.693 | val loss 0.8024 bal_acc 0.819 f1 0.548
Epoch 05/10 | train loss 0.2289 bal_acc 0.954 f1 0.761 | val loss 1.0421 bal_acc 0.826 f1 0.623
Epoch 06/10 | train loss 0.1611 bal_acc 0.970 f1 0.830 | val loss 1.2310 bal_acc 0.812 f1 0.535
Epoch 07/10 | train loss 0.1510 bal_acc 0.974 f1 0.838 | val loss 1.2524 bal_acc 0.813 f1 0.586
Epoch 08/10 | train loss 0.0897 bal_acc 0.987 f1 0.902 | val loss 1.9190 bal_acc 0.781 f1 0.607
Epoch 09/10 | train loss 0.0798 bal_acc 0.986 f1 0.910 | val loss 1.6586 bal_acc 0.823 f1 0.649
Epoch 10/10 | train loss 0.0652 bal_acc 0.990 f1 0.928 | val loss 1.6218 bal_acc 0.813 f1 0.626

Best m

## 6. Run Baseline

In [6]:
!python baseline.py

Baseline (TF-IDF + LinearSVC) test accuracy: 0.927

Balanced accuracy: 0.849; toxic-class F1: 0.655

              precision    recall  f1-score   support

        safe       0.97      0.94      0.96      1939
       toxic       0.58      0.75      0.65       195

    accuracy                           0.93      2134
   macro avg       0.78      0.85      0.81      2134
weighted avg       0.94      0.93      0.93      2134



## 7. Save Checkpoint And Report Evidence To Drive

The repository already lives in Drive if `PROJECT_DIR` points there. This cell also copies the selected weights and small result files into an explicit Colab artifacts folder.

In [7]:
from pathlib import Path
import shutil

artifact_dir = Path(PROJECT_DIR) / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

checkpoint = Path('checkpoints/best_model.pt')
if checkpoint.exists():
    shutil.copy2(checkpoint, artifact_dir / checkpoint.name)

results_dir = Path('results')
for result in results_dir.glob('*'):
    if result.is_file():
        shutil.copy2(result, artifact_dir / result.name)

print(f'Copied checkpoint and result evidence to {artifact_dir}')

Copied checkpoint and result evidence to /content/drive/MyDrive/APS360/Final_Project/artifacts
